In [1]:
!pip install mediapipe opencv-python

In [1]:
pip cache purge

Files removed: 6 (250 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade pip wheel setuptools

Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install opencv-python
!pip install mediapipe

In [ ]:
# import cv2
# import mediapipe as mp
# import numpy as np

: 

: 

In [ ]:
# mp_face_mesh = mp.solutions.face_mesh
# mp_drawing = mp.solutions.drawing_utils
# mp_pose = mp.solutions.pose

: 

: 

In [ ]:
# cap = cv2.VideoCapture(0)
# #change accuracy later
# # Set up Face Mesh
# # 'max_num_faces' can be adjusted based on your needs
# face_mesh = mp_face_mesh.FaceMesh(
#     min_detection_confidence=0.5,
#     min_tracking_confidence=0.5,
#     max_num_faces=1
# )
# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
#     while cap.isOpened():
#         ret,frame = cap.read()

#         #recolour the image
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         #main part ie making detections
#         results = pose.process(image)

#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) #for opencv

#         #landmarks has coordinates of each individual 
#         #pose connections has all connections between joints and parts of body
#         #last arg is for connection lines
#         #last but one is for landmarks
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
#                                   mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
#                                   mp_drawing.DrawingSpec(color=(254,66,230), thickness=2, circle_radius=2 ))

#         cv2.imshow('Camera Feed', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

#     cap.release()
#     cv2.destroyAllWindows()

: 

: 

In [ ]:
# results.pose_landmarks

: 

: 

In [ ]:
#mp_pose.POSE_CONNECTIONS

: 

: 

In [4]:
import cv2
import mediapipe as mp
import numpy as np

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
mp_face_mesh = mp.solutions.face_mesh

cap = cv2.VideoCapture(0)

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    max_num_faces=1
)

pose = mp_pose.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    enable_segmentation=True
)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    image = cv2.cvtColor(cv2.flip(frame, 1), cv2.COLOR_BGR2RGB)
    image.flags.writeable = False

    results_face = face_mesh.process(image)
    results_pose = pose.process(image)

    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    h, w, _ = image.shape

    if results_face.multi_face_landmarks:
        for face_landmarks in results_face.multi_face_landmarks:
            # Draw the face mesh
            mp_drawing.draw_landmarks(
                image,
                face_landmarks,
                mp_face_mesh.FACEMESH_CONTOURS,
                mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=1, circle_radius=1),
                mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=1, circle_radius=1)
            )

            # Face dimension calculation code
            landmarks = face_landmarks.landmark
            def get_pixel_distance(p1, p2):
                x1, y1 = int(p1.x * w), int(p1.y * h)
                x2, y2 = int(p2.x * w), int(p2.y * h)
                return ((x2 - x1)**2 + (y2 - y1)**2)**0.5
            # Body dimension calculation
            def get_pixel_distance2(p1, p2, width, height):
                x1, y1 = int(p1.x * width), int(p1.y * height)
                x2, y2 = int(p2.x * width), int(p2.y * height)
                return ((x2 - x1)**2 + (y2 - y1)**2)**0.5

            # Eyes
            left_eye_width = get_pixel_distance(landmarks[133], landmarks[33])
            right_eye_width = get_pixel_distance(landmarks[362], landmarks[263])
            cv2.putText(image, f"Left Eye: {left_eye_width:.2f}px", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
            cv2.putText(image, f"Right Eye: {right_eye_width:.2f}px", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

            # Lips
            mouth_width = get_pixel_distance(landmarks[61], landmarks[291])
            mouth_height = get_pixel_distance(landmarks[0], landmarks[17])
            cv2.putText(image, f"Mouth Width: {mouth_width:.2f}px", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
            cv2.putText(image, f"Mouth Height: {mouth_height:.2f}px", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

            # Cheekbones
            cheek_width = get_pixel_distance(landmarks[205], landmarks[425])
            cv2.putText(image, f"Cheekbone Width: {cheek_width:.2f}px", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)


    # Drawing Body Pose Landmarks and Calculating Dimensions
    if results_pose.pose_landmarks:
        mp_drawing.draw_landmarks(
            image,
            results_pose.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(254, 66, 230), thickness=2, circle_radius=2)
        )
        
        # Calculate and display body dimensions
        landmarks_pose = results_pose.pose_landmarks.landmark

        # Shoulder Width
        shoulder_width = get_pixel_distance2(landmarks_pose[mp_pose.PoseLandmark.LEFT_SHOULDER.value], 
                                            landmarks_pose[mp_pose.PoseLandmark.RIGHT_SHOULDER.value], w, h)
        cv2.putText(image, f"Shoulder Width: {shoulder_width:.2f}px", (10, 200), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Hip Width
        hip_width = get_pixel_distance2(landmarks_pose[mp_pose.PoseLandmark.LEFT_HIP.value],
                                       landmarks_pose[mp_pose.PoseLandmark.RIGHT_HIP.value], w, h)
        cv2.putText(image, f"Hip Width: {hip_width:.2f}px", (10, 230), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        # Torso Length (approximation)
        shoulder_midpoint_y = (landmarks_pose[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y + landmarks_pose[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y) / 2
        hip_midpoint_y = (landmarks_pose[mp_pose.PoseLandmark.LEFT_HIP.value].y + landmarks_pose[mp_pose.PoseLandmark.RIGHT_HIP.value].y) / 2
        torso_length = abs(int((shoulder_midpoint_y - hip_midpoint_y) * h))
        cv2.putText(image, f"Torso Length: {torso_length:.2f}px", (10, 260),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                    
        # Left Arm Length
        left_arm_length = get_pixel_distance2(landmarks_pose[mp_pose.PoseLandmark.LEFT_SHOULDER.value], 
                                             landmarks_pose[mp_pose.PoseLandmark.LEFT_WRIST.value], w, h)
        cv2.putText(image, f"Left Arm Length: {left_arm_length:.2f}px", (10, 290), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                    
        # Left Leg Length
        left_leg_length = get_pixel_distance2(landmarks_pose[mp_pose.PoseLandmark.LEFT_HIP.value],
                                             landmarks_pose[mp_pose.PoseLandmark.LEFT_ANKLE.value], w, h)
        cv2.putText(image, f"Left Leg Length: {left_leg_length:.2f}px", (10, 320),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        # Right Arm Length
        right_arm_length = get_pixel_distance2(landmarks_pose[mp_pose.PoseLandmark.RIGHT_SHOULDER.value], 
                                              landmarks_pose[mp_pose.PoseLandmark.RIGHT_WRIST.value], w, h)
        cv2.putText(image, f"Right Arm Length: {right_arm_length:.2f}px", (10, 350), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        # Right Leg Length
        right_leg_length = get_pixel_distance2(landmarks_pose[mp_pose.PoseLandmark.RIGHT_HIP.value],
                                              landmarks_pose[mp_pose.PoseLandmark.RIGHT_ANKLE.value], w, h)
        cv2.putText(image, f"Right Leg Length: {right_leg_length:.2f}px", (10, 380),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
 
        results = results_pose
        if results.pose_landmarks and results.segmentation_mask is not None:
            landmarks = results.pose_landmarks.landmark
            h, w, _ = image.shape

            #  1. HEIGHT 
            left_eye = landmarks[mp_pose.PoseLandmark.LEFT_EYE.value]
            right_eye = landmarks[mp_pose.PoseLandmark.RIGHT_EYE.value]
            left_heel = landmarks[mp_pose.PoseLandmark.LEFT_HEEL.value]
            right_heel = landmarks[mp_pose.PoseLandmark.RIGHT_HEEL.value]
            head_y = int((left_eye.y + right_eye.y) * 0.5 * h)
            feet_y = int((left_heel.y + right_heel.y) * 0.5 * h)
            pixel_height = abs(head_y - feet_y)
            cv2.putText(image, f"Pixel Height: {pixel_height} px", (900, 150), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

            #  2. HIP WIDTH 
            left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]
            right_hip = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value]
            pixel_hip_width = abs(int(left_hip.x * w - right_hip.x * w))
            cv2.putText(image, f"Pixel Hip Width: {pixel_hip_width} px", (900, 190), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

            # 3. MUAC (ARM THICKNESS) 
            shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
            elbow = landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value]
            z_diff = abs(shoulder.z - elbow.z)
            min_z_diff = 0.2

            cv2.putText(image, f"Z Difference: {z_diff:.2f}", (900, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

            if z_diff > min_z_diff:
                cv2.putText(image, "Status: Keep arm parallel", (900, 70), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            else:
                cv2.putText(image, "Status: Arm is Parallel", (900, 70), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                
                segmentation_mask = results.segmentation_mask
                if segmentation_mask is not None:
                    resized_mask = cv2.resize(segmentation_mask, (w, h))
                    binary_mask = np.stack((resized_mask,) * 3, axis=-1) > 0.1
                    
                    shoulder_xy = (int(shoulder.x * w), int(shoulder.y * h))
                    elbow_xy = (int(elbow.x * w), int(elbow.y * h))
                    midpoint_x = int((shoulder_xy[0] + elbow_xy[0]) / 2)
                    midpoint_y = int((shoulder_xy[1] + elbow_xy[1]) / 2)
                    
                    if midpoint_y < h and midpoint_x < w and binary_mask[midpoint_y, midpoint_x, 0]:
                        arm_pixel_thickness = 0
                        y = midpoint_y
                        while y > 0 and binary_mask[y, midpoint_x, 0]:
                            arm_pixel_thickness += 1; y -= 1
                        y = midpoint_y + 1
                        while y < h and binary_mask[y, midpoint_x, 0]:
                            arm_pixel_thickness += 1; y += 1
                        
                        cv2.circle(image, (midpoint_x, midpoint_y), 7, (0, 255, 0), -1)
                        cv2.putText(image, f"Pixel Arm Thickness: {arm_pixel_thickness} px", (900, 110), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0, 0), 2)
                    else:
                        cv2.putText(image, "Error: Midpoint is off mask", (900, 110), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    

    cv2.imshow('MediaPipe Body and Face Tracking', image)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()

# Printing Landmark 
if 'results_pose' in locals() and results_pose.pose_landmarks:
    landmarks_pose = results_pose.pose_landmarks.landmark
    print("\n--- Body Pose Landmarks ---")
    print("Number of body landmarks:", len(landmarks_pose))
    print("Nose:", landmarks_pose[mp_pose.PoseLandmark.NOSE.value])
    print("Right Hip:", landmarks_pose[mp_pose.PoseLandmark.RIGHT_HIP.value])
    print("Left Ankle:", landmarks_pose[mp_pose.PoseLandmark.LEFT_ANKLE.value])

if 'results_face' in locals() and results_face.multi_face_landmarks:
    landmarks_face = results_face.multi_face_landmarks[0].landmark
    print("\n--- Facial Landmarks ---")
    print("Number of facial landmarks:", len(landmarks_face))
    # Print a few examples
    print("Nose Tip (landmark 1):", landmarks_face[1])
    print("Mouth Left Corner (landmark 61):", landmarks_face[61])
    print("Right Eye (landmark 263):", landmarks_face[263])

/Users/vaishnukanna/ml/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
2025-10-15 14:00:58.491 Python[22873:16213316] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.
I0000 00:00:1760517059.833643 16213316 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
I0000 00:00:1760517059.844978 16213316 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1760517059.864076 16213934 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1760517059.868060 


--- Body Pose Landmarks ---
Number of body landmarks: 33
Nose: x: 0.420641124
y: 0.574507713
z: -0.863157392
visibility: 0.999871612

Right Hip: x: 0.464005977
y: 1.92375493
z: 0.105074249
visibility: 0.000117778683

Left Ankle: x: 0.863140881
y: 3.40969348
z: 0.24323374
visibility: 2.07529356e-05


--- Facial Landmarks ---
Number of facial landmarks: 468
Nose Tip (landmark 1): x: 0.410988331
y: 0.628919303
z: -0.0556818508

Mouth Left Corner (landmark 61): x: 0.382069767
y: 0.726742089
z: 0.0112196384

Right Eye (landmark 263): x: 0.486898482
y: 0.493044943
z: 0.00805032067



In [ ]:
# results_face = face_mesh.process(image)

# if results_face.multi_face_landmarks:
#     landmarks = results_face.multi_face_landmarks[0].landmark
#     print("Number of facial landmarks:", len(landmarks))

#     # Print a few example landmarks by their index
#     print("\nExample Facial Landmarks:")
#     print("Nose Tip (landmark 1):", landmarks[1])
#     print("Left Eye (landmark 33):", landmarks[33])
#     print("Right Eye (landmark 263):", landmarks[263])
#     print("Mouth Left (landmark 61):", landmarks[61])
#     print("Mouth Right (landmark 291):", landmarks[291])

#     # To get all landmark indices, you can't use an Enum like with Pose.
#     # The landmarks are simply indexed from 0 to 467.
#     print("\nIterating through all facial landmarks:")
#     for i, landmark in enumerate(landmarks):
#         print(f"Landmark {i}: x={landmark.x:.4f}, y={landmark.y:.4f}, z={landmark.z:.4f}")

: 

: 

In [ ]:
# landmarks = results_pose.pose_landmarks.landmark
# print(len(landmarks))
# for lm in mp_pose.PoseLandmark:
#     print(lm.value, ":", lm)

: 

: 

In [ ]:
# mp_pose.PoseLandmark.NOSE.value

: 

: 

In [ ]:
# landmarks[mp_pose.PoseLandmark.NOSE.value]

: 

: 